In [0]:
#### PRODUCT
##### CREATED BY    : Pratik J
##### CREATED DATE  : 20260817
##### DISCRIPTION   : Product master data containing product identifiers, category,description, dimensions, weight, and product image information. The Silver layer standardizes product attributes and validates product master data for downstream product and sales analytics.
##### MODIFIED BY   : Pratik J
##### MODIFIED DATE : 20260825

#####- Standardize column names to snake_case
#####-- Trim string columns
#####-- Standardize product category names
#####-- Replace underscores with spaces where required
#####-- Cast numeric attributes to integer
#####-- Validate product dimensions
#####-- Validate product weight
#####-- Remove duplicate products
#####-- Handle mandatory-field NULL values
#####-- Add audit timestamp `ingest_ts`

READ CSV

In [0]:
%run "/Workspace/Users/pjadhav564@gmail.com/Brazil_project/functions"

In [0]:
path = "/Volumes/e_commerce_brazil/e_com_bronze/bronze_clean/"
product_df = read_csv(f"{path}/products")


print(product_df.columns)

In [0]:
#data cleaning and validation 

product_df = product_df.withColumnRenamed("PRODid", "product_id") \
    .withColumnRenamed("productcattegory_name", "product_category_name") \
    .withColumnRenamed("productnamlenght", "product_name_length") \
    .withColumnRenamed("product_description_lenght", "product_description_length") \
    .withColumnRenamed("productphotosqty", "product_photos_qty") \
    .withColumnRenamed("productweight_g", "product_weight_g") \
    .withColumnRenamed("prodlenghth", "product_length_cm")

product_df = clean_column_names(product_df)
product_df = trim_space_col(product_df,["product_id","product_category_name","product_name_length","product_description_length","product_photos_qty","product_weight_g","product_length_cm","product_height_cm","product_width_cm"])

product_df = cast_col(product_df,{"product_name_length":"integer","product_description_length":"integer","product_photos_qty":"integer","product_weight_g":"integer","product_length_cm":"integer","product_height_cm":"integer","product_width_cm":"integer"})

product_df = drop_duplicates(product_df,"product_id")
product_df.printSchema()
product_df = remove_null(product_df,["product_id"])
product_df = handle_null(product_df,{"product_category_name":"unknown","product_name_length":0,"product_description_length":0,"product_photos_qty":0,"product_weight_g":0,"product_length_cm":0,"product_height_cm":0,"product_width_cm":0})
product_df = product_df.withColumn("product_category_name",regexp_replace(col("product_category_name"), "_", " "))

product_df = product_df.filter((col("product_name_length")>=0) & (col("product_description_length")>=0) & (col("product_photos_qty")>=0) & (col("product_weight_g")>=0) & (col("product_length_cm")>=0) & (col("product_height_cm") >= 0) & (col("product_width_cm")>=0))

product_df = update_ts(product_df)
                         
display(product_df)

WRITE CSV

In [0]:
silver_path = "/Volumes/e_commerce_brazil/e_com_silver/updated_silver"
writefile(product_df,f'{silver_path}/product_silver',"overwrite")